# Week 12 Activity: Text as Data and LLMs

# Setup


In [1]:
## imports
import pandas as pd
import numpy as np
import re

## nltk imports
import nltk
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
_ = nltk.download(['punkt', 'punkt_tab', 'stopwords'], quiet = True)

## spacy for named entity recognition --- if you get an error,
## run this in terminal: python -m spacy download en_core_web_sm
import spacy
nlp = spacy.load('en_core_web_sm')

## sentiment analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## calling LLM APIs
import requests
import yaml
import time

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
from gensim import corpora
import gensim

## visualization - if you have issues installing, comment out
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rj545/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Setup code for the Airbnb listings 

In [2]:
## load data (path relative to the activities folder)
ab = pd.read_csv("../datasets/airbnb_text.zip")
ab.head()

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80


In [19]:
## stopword list from the slides: nltk english stopwords + custom words
list_stopwords = stopwords.words("english")
custom_words_toadd = ['apartment', 'new york', 'nyc',
                      'bronx', 'brooklyn',
                      'manhattan', 'queens',
                      'staten island']
list_stopwords_new = list_stopwords + custom_words_toadd

## lowercase corpus
corpus_lower = ab.name.str.lower().to_list()

## initialize stemmer
porter = PorterStemmer()

In [4]:
## document-term matrix function from the slides
def create_dtm(list_of_strings, metadata):
    vectorizer = CountVectorizer(lowercase = True)
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(),
                                   columns = vectorizer.get_feature_names_out())
    metadata = metadata.copy()
    metadata.columns = ["metadata_" + col for col in metadata.columns]
    dtm_dense_named_withid = pd.concat([metadata.reset_index(),
                                        dtm_dense_named], axis = 1)
    return dtm_dense_named_withid

In [5]:
## random sample of 1000 listings from the slides
ab_small = (ab.loc[~ab.name.isnull(),
                   ['id', 'neighbourhood_group', 'price', 'name']]
            .copy()
            .rename(columns = {'price': 'price_rawdata'})
            .sample(n = 1000, random_state = 9899))
ab_small['name_lower'] = ab_small['name'].str.lower()
ab_small.head()

,id,neighbourhood_group,price_rawdata,name,name_lower
22540,18227529,Manhattan,140,Live in New York Near Central Park and Columbi...,live in new york near central park and columbi...
47531,35794273,Brooklyn,130,Hope Garden,hope garden
33906,26858196,Brooklyn,75,AWESOME 2 BEDS - QUEEN + SOFA - NEXT TO METRO,awesome 2 beds - queen + sofa - next to metro
12047,9369514,Brooklyn,25,STARTUP CHEAP PLACE BROOKLYN,startup cheap place brooklyn
2908,1669149,Manhattan,250,Beautiful Modern Midtown Apartment,beautiful modern midtown apartment


# Activity 1: named entity recognition

Play around with different variations of the tweet below and look at the results. E.g.:

- What happens if you remove "in the year 2000"?
- What if you make AOL lowercase?
- What if you add Kara Swisher's name?
- What if you add that she's a journalist?

In [10]:
## tweet from the slides (modified from a real tweet)
m_tweet = """When AOL was founded in the year 2000 I remember raising my hand
and asking 'well...if you're using my data - where is my $50'
and no one had a good answer...and there still isn't one.
"""

In [11]:
## helper function to print the named entities in a string
def print_ents(one_string):
    for one_tok in nlp(one_string).ents:
        print("Entity: " + one_tok.text + "; NER tag: " + one_tok.label_)

## original tweet
print_ents(m_tweet)

Entity: AOL; NER tag: ORG
Entity: the year 2000; NER tag: DATE
Entity: 50; NER tag: MONEY


In [12]:
## A. remove "in the year 2000"
m_tweet_no2000 = m_tweet.replace("in the year 2000", "")
print_ents(m_tweet_no2000)

Entity: AOL; NER tag: ORG
Entity: 50; NER tag: MONEY


In [13]:
## B. make AOL lowercase
m_tweet_aol = m_tweet.replace("AOL", "aol")
print_ents(m_tweet_aol)

Entity: the year 2000; NER tag: DATE
Entity: 50; NER tag: MONEY


In [16]:
## C. add Kara Swisher's name
m_tweet_ks = m_tweet.replace("asking", "asking journalist Kara Swisher")
print_ents(m_tweet_ks)

Entity: AOL; NER tag: ORG
Entity: the year 2000; NER tag: DATE
Entity: Kara Swisher; NER tag: PERSON
Entity: 50; NER tag: MONEY


In [ ]:
## D. add that she's a journalist


# Activity 2: preprocess all the listings

In the slides, we performed preprocessing on a single Airbnb listing. We want to generalize this preprocessing across all listings.

- Embed step two (remove stopwords) and step three (stem) into one or two functions that take in a raw string (e.g., the raw text of an Airbnb listing) and return a preprocessed string
- Apply the function to all the texts in `corpus_lower` created in the setup code block above

In [25]:
## your code here
def process_step1(one_str):
    try:
        ## remove stopwords
        nostop_listing1 = [word for word in wordpunct_tokenize(one_str)
                           if word not in list_stopwords_new]
        ## stem, keeping only alphabetic tokens with 3+ characters
        clean_listing = [porter.stem(word) for word in nostop_listing1
                         if word.isalpha()
                         and len(word) > 2]
        clean_listing_str = " ".join(clean_listing)
        return clean_listing_str
    except:
        ## missing listings (NaN) aren't strings
        return ""

In [26]:
corpus_lower[0:5]


['clean & quiet apt home by the park',
 'skylit midtown castle',
 'the village of harlem....new york !',
 'cozy entire floor of brownstone',
 'entire apt: spacious studio/loft by central park']

In [27]:
clean_listings = [process_step1(one_text) for one_text in corpus_lower]


['clean quiet apt home park',
 'skylit midtown castl',
 'villag harlem new york',
 'cozi entir floor brownston',
 'entir apt spaciou studio loft central park']

In [28]:
clean_listings[0:10]

['clean quiet apt home park',
 'skylit midtown castl',
 'villag harlem new york',
 'cozi entir floor brownston',
 'entir apt spaciou studio loft central park',
 'larg cozi midtown east',
 'blissartsspac',
 'larg furnish room near way',
 'cozi clean guest room famili apt',
 'cute cozi lower east side bdrm']

# Activity 3: document-term matrix with the preprocessed text

- Stick with the same random sample of 1000 (`ab_small`)
- Apply the preprocessing steps from Activity 2 to create a new column in `ab_small` with the preprocessed text (if you got stuck on that, try just removing stopwords)
- Use the `create_dtm` function to create a document-term matrix from the preprocessed data
- Take the sum of each of the term columns to find the top words

In [ ]:
## your code here

In [31]:
ab_small['processed'] = [process_step1(one_text) for one_text in 
                        ab_small.name.str.lower().to_list()]
ab_small.head()

metadata_list = ['id', 'neighbourhood_group', 'price_rawdata']
dtm_pre = create_dtm(list_of_strings = ab_small.processed,
                     metadata = ab_small[metadata_list])

dtm_pre.head()

,id,neighbourhood_group,price_rawdata,name,name_lower,processed
22540,18227529,Manhattan,140,Live in New York Near Central Park and Columbi...,live in new york near central park and columbi...,live new york near central park columbia
47531,35794273,Brooklyn,130,Hope Garden,hope garden,hope garden
33906,26858196,Brooklyn,75,AWESOME 2 BEDS - QUEEN + SOFA - NEXT TO METRO,awesome 2 beds - queen + sofa - next to metro,awesom bed queen sofa next metro
12047,9369514,Brooklyn,25,STARTUP CHEAP PLACE BROOKLYN,startup cheap place brooklyn,startup cheap place
2908,1669149,Manhattan,250,Beautiful Modern Midtown Apartment,beautiful modern midtown apartment,beauti modern midtown


,index,metadata_id,metadata_neighbourhood_group,metadata_price_rawdata,abod,access,across,ador,afford,airbnb,...,young,zen,交通便利,位于北上远离开辆,家庭式獨立衛生間套房g,温馨小筑,简单的四房一厅两卫生间,纽约之家,走路四分钟到地铁站,따뜻한
0,22540,18227529,Manhattan,140,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,47531,35794273,Brooklyn,130,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,33906,26858196,Brooklyn,75,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,12047,9369514,Brooklyn,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2908,1669149,Manhattan,250,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
dtm_pre_termsonly = dtm_pre[[col for col in dtm_pre.columns if
                             "metadata" not in col and col != "index"]]

dtm_pre_termsonly.sum().sort_values(ascending = False).head(20)

room            208
bedroom         167
privat          146
cozi            134
apt              97
studio           94
spaciou          83
park             79
sunni            66
east             64
near             56
beauti           52
bed              50
central          47
williamsburg     47
home             44
villag           44
bright           42
luxuri           41
heart            40
dtype: int64

# Activity 4: topic model with the preprocessed text

- Repeat the steps for estimating the topic model, this time with the preprocessed texts
- Play around with other parameters like the number of topics and the filtering bounds --- what seems to produce useful topics?
- Explore the average listing price for each topic 

In [ ]:
## your code here

# Activity 5: LLMs for airbnb listing classification

- A. Load your credentials and read through the function `ask_llm` that takes in a prompt and returns the model's text response
- B. Run the code to create a random sample of 20 airbnb listings - `ab_llm` - change the seed to select a different sample
- C. Feed `ask_llm` a prompt to classify a listing as a "budget", "standard", or "luxury" stay. 
- D. Does the mean price differ across the LLM's categories?

## A. Credentials and function

In [3]:
## load creds
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

gemini_key = creds['gemini_api']['api_key']

## Gemini's OpenAI-compatible endpoint
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

## model to use --- see https://ai.google.dev/gemini-api/docs/models
## for current free-tier model names
gemini_model = "gemini-3.5-flash-lite"

In [4]:
def ask_llm(question_text, api_key, model = gemini_model, url = gemini_url,
            max_tries = 5):
    """
    Query an LLM and return a response
    Inputs: 
    question_text: text of a prompt
    api_key: api key for Gemini
    model: gemini model (defaults to 3.5 flash lite)
    url: url for Gemini
    max_tries: how many times to attempt to query in case of error
    Returns:
    content of LLM response 
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    ## temperature = 0 makes the output (close to) deterministic
    data = {
        "model": model,
        "temperature": 0,
        ## gemini 3 models "think" before answering; low effort keeps
        ## simple classification tasks fast (thinking can't be turned off)
        "reasoning_effort": "low",
        "messages": [{"role": "user", "content": question_text}]
    }
    for attempt in range(max_tries):
        response = requests.post(url, json = data,
                                 headers = headers, timeout = 120)
        ## success: return the text of the response
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content'].strip()
        ## model overloaded (503) or rate limit hit (429): wait and retry,
        ## doubling the wait each time (2, 4, 8, ... seconds)
        if response.status_code in [429, 503]:
            wait = 2 ** (attempt + 1)
            print(f"API busy (error {response.status_code}); retrying in {wait} seconds")
            time.sleep(wait)
        else:
            ## other errors (e.g., bad key): stop and show the message
            raise Exception(f"API error {response.status_code}: {response.text}")
    raise Exception(f"API still unavailable after {max_tries} tries; try again later")


## test
ask_llm("In one sentence, what is a large language model?", gemini_key)

'A large language model is an advanced artificial intelligence system trained on vast amounts of text data to understand, summarize, generate, and converse in human-like language.'

## B. Randomly sample a subset of listings

In [5]:
ab_llm = (ab.loc[~ab.name.isnull(), ['id', 'name', 'neighbourhood_group', 'price']]
          .sample(n = 20, random_state = 60525)
          .reset_index(drop = True))

## C and D. Prompt to classify the listing as budget, standard, or luxury

In [7]:
## your code here
ab_llm.head()
tier_prompt = ("Here is an Airbnb listing from New York City"
              "Based on the title of this listing, classify the listing as either"
              "budget, standard, or luxury. Answer with only one word: budget, standard"
              "or luxury. Here is the listing title: ")


,id,name,neighbourhood_group,price
0,11101754,Upscale Sun-Soaked Luxury apartment in Brooklyn,Brooklyn,145
1,5251673,Quiet Space Directly Off L Train,Brooklyn,115
2,35573040,Summer sublet in south slope,Brooklyn,109
3,35934299,Penthouse large 1 BR with huge terrace (7C),Brooklyn,195
4,2687375,Queen Size Foam Mattress,Manhattan,70


In [9]:
tier_output = []
for one_listing in ab_llm.name:
    try:
        tier_output.append(ask_llm(tier_prompt + one_listing, gemini_key).lower())
    except Exception as e:
        print(e)
        tier_output.append(np.nan)
    time.sleep(4) # stay under the free tier's requests-per-minute limit

In [11]:
ab_llm['tier_llm'] = tier_output
ab_llm[['name', 'price', 'tier_llm']]

,name,price,tier_llm
0,Upscale Sun-Soaked Luxury apartment in Brooklyn,145,luxury
1,Quiet Space Directly Off L Train,115,standard
2,Summer sublet in south slope,109,standard
3,Penthouse large 1 BR with huge terrace (7C),195,luxury
4,Queen Size Foam Mattress,70,budget
5,Brooklyn Cove Studio Apt w/ Garden!!,65,standard
6,Spacious 2-bedroom townhouse apt in Clinton Hill,225,standard
7,Cozy Oversized Room - near Park Ave,80,standard
8,Beautiful 1 bedroom apartment,120,standard
9,Perfect West Village Apt,195,luxury


In [13]:
ab_llm.groupby('tier_llm').agg(mean_price = ('price', 'mean'),
                               n_listings = ('price', 'size'))

,mean_price,n_listings
tier_llm,,
budget,81.666667,3
luxury,197.166667,6
standard,102.272727,11
